Question 1: Impact of Preprocessing on Model Performance

A company wants to predict employee salaries using age, experience, and department. The features have different scales, and department is categorical.

import pandas as pd

df = pd.DataFrame({
    "Age": [22, 25, 28, 30, 35, 40, 45, 50, 55, 60],
    "Experience": [0, 2, 4, 6, 10, 15, 20, 25, 30, 35],
    "Department": [
        "IT", "HR", "IT", "Sales", "HR",
        "IT", "Sales", "IT", "HR", "Sales"
    ],
    "Salary": [
        25000, 28000, 32000, 35000, 42000,
        55000, 60000, 75000, 80000, 95000
    ]
})
A. Build two regression models using LinearRegression: one with the numerical features unchanged and categorical data encoded using OneHotEncoder, and another with numerical features scaled using StandardScaler. Use the same train-test split and compare MAE and \(R^2\). Explain why scaling can affect some algorithms more than others.



In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.DataFrame({
    "Age": [22, 25, 28, 30, 35, 40, 45, 50, 55, 60],
    "Experience": [0, 2, 4, 6, 10, 15, 20, 25, 30, 35],
    "Department": [
        "IT", "HR", "IT", "Sales", "HR",
        "IT", "Sales", "IT", "HR", "Sales"
    ],
    "Salary": [
        25000, 28000, 32000, 35000, 42000,
        55000, 60000, 75000, 80000, 95000
    ]
})

X = df[["Age", "Experience", "Department"]]
y = df["Salary"]

numeric_features = ["Age", "Experience"]
categorical_features = ["Department"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

model_unscaled = Pipeline([
    ("preprocessor", preprocessor_unscaled),
    ("regressor", LinearRegression())
])

model_scaled = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("regressor", LinearRegression())
])

model_unscaled.fit(X_train, y_train)
model_scaled.fit(X_train, y_train)

pred_unscaled = model_unscaled.predict(X_test)
pred_scaled = model_scaled.predict(X_test)
results = pd.DataFrame({
    "Model": ["Linear Regression - Unscaled",
              "Linear Regression - Scaled"],
    "MAE": [
        mean_absolute_error(y_test, pred_unscaled),
        mean_absolute_error(y_test, pred_scaled)
    ],
    "R2": [
        r2_score(y_test, pred_unscaled),
        r2_score(y_test, pred_scaled)
    ]
})

print("Test data:")
print(X_test)
print("\nActual salaries:")
print(y_test.to_list())

print("\nModel comparison:")
print(results.round(4).to_string(index=False))


Test data:
   Age  Experience Department
8   55          30         HR
1   25           2         HR

Actual salaries:
[80000, 28000]

Model comparison:
                       Model       MAE     R2
Linear Regression - Unscaled 2601.0838 0.9861
  Linear Regression - Scaled 2601.0838 0.9861


B. Replace LinearRegression with KNeighborsRegressor and repeat the comparison. Explain how differences in feature scales can affect distance calculations and influence which employees are considered similar.

In [2]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.DataFrame({
    "Age": [22, 25, 28, 30, 35, 40, 45, 50, 55, 60],
    "Experience": [0, 2, 4, 6, 10, 15, 20, 25, 30, 35],
    "Department": [
        "IT", "HR", "IT", "Sales", "HR",
        "IT", "Sales", "IT", "HR", "Sales"
    ],
    "Salary": [
        25000, 28000, 32000, 35000, 42000,
        55000, 60000, 75000, 80000, 95000
    ]
})

X = df[["Age", "Experience", "Department"]]
y = df["Salary"]

numeric_features = ["Age", "Experience"]
categorical_features = ["Department"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor_unscaled = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

preprocessor_scaled = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

model_unscaled = Pipeline([
    ("preprocessor", preprocessor_unscaled),
    ("regressor", KNeighborsRegressor(n_neighbors=3))
])

model_scaled = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("regressor", KNeighborsRegressor(n_neighbors=3))
])

model_unscaled.fit(X_train, y_train)
model_scaled.fit(X_train, y_train)

pred_unscaled = model_unscaled.predict(X_test)
pred_scaled = model_scaled.predict(X_test)

results = pd.DataFrame({
    "Model": ["KNN - Unscaled", "KNN - Scaled"],
    "MAE": [
        mean_absolute_error(y_test, pred_unscaled),
        mean_absolute_error(y_test, pred_scaled)
    ],
    "R2": [
        r2_score(y_test, pred_unscaled),
        r2_score(y_test, pred_scaled)
    ]
})

print("Actual salaries:", y_test.to_list())
print("Unscaled predictions:", pred_unscaled)
print("Scaled predictions:", pred_scaled)

print("\nModel comparison:")
print(results.round(4).to_string(index=False))


Actual salaries: [80000, 28000]
Unscaled predictions: [76666.66666667 30666.66666667]
Scaled predictions: [76666.66666667 33000.        ]

Model comparison:
         Model       MAE     R2
KNN - Unscaled 3000.0000 0.9865
  KNN - Scaled 4166.6667 0.9733


Question 2: ColumnTransformer for Mixed Data
A retail company wants to predict customer spending using numerical and categorical features.

import pandas as pd

df = pd.DataFrame({
    "Age": [21, 25, 30, 35, 40, 45, 50, 28, 33, 38],
    "Annual_Income": [
        20000, 25000, 30000, 35000, 45000,
        50000, 60000, 28000, 40000, 48000
    ],
    "City": [
        "Delhi", "Mumbai", "Bhopal", "Delhi", "Pune",
        "Mumbai", "Bhopal", "Pune", "Delhi", "Mumbai"
    ],
    "Membership": [
        "Basic", "Silver", "Gold", "Basic", "Gold",
        "Silver", "Gold", "Basic", "Silver", "Gold"
    ],
    "Spending": [
        2000, 3000, 4500, 4000, 6000,
        6500, 8000, 3500, 5000, 7000
    ]
})
A. Create a ColumnTransformer that applies StandardScaler to Age and Annual_Income, and OneHotEncoder(handle_unknown="ignore") to City and Membership. Display the transformed feature matrix and identify how many output features are created.



In [3]:

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.DataFrame({
    "Age": [21, 25, 30, 35, 40, 45, 50, 28, 33, 38],
    "Annual_Income": [
        20000, 25000, 30000, 35000, 45000,
        50000, 60000, 28000, 40000, 48000
    ],
    "City": [
        "Delhi", "Mumbai", "Bhopal", "Delhi", "Pune",
        "Mumbai", "Bhopal", "Pune", "Delhi", "Mumbai"
    ],
    "Membership": [
        "Basic", "Silver", "Gold", "Basic", "Gold",
        "Silver", "Gold", "Basic", "Silver", "Gold"
    ],
    "Spending": [
        2000, 3000, 4500, 4000, 6000,
        6500, 8000, 3500, 5000, 7000
    ]
})

X = df[["Age", "Annual_Income", "City", "Membership"]]

numeric_features = ["Age", "Annual_Income"]
categorical_features = ["City", "Membership"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

transformed_df = pd.DataFrame(
    X_transformed,
    columns=feature_names,
    index=df.index
)

print("Transformed Feature Matrix:")
print(transformed_df.round(3))

print("\nOutput feature names:")
print(list(feature_names))

print("\nTotal output features:", len(feature_names))
print("Transformed matrix shape:", X_transformed.shape)


Transformed Feature Matrix:
   num__Age  num__Annual_Income  cat__City_Bhopal  cat__City_Delhi  \
0    -1.580              -1.505               0.0              1.0   
1    -1.112              -1.089               0.0              0.0   
2    -0.527              -0.673               1.0              0.0   
3     0.059              -0.258               0.0              1.0   
4     0.644               0.574               0.0              0.0   
5     1.229               0.989               0.0              0.0   
6     1.814               1.821               1.0              0.0   
7    -0.761              -0.840               0.0              0.0   
8    -0.176               0.158               0.0              1.0   
9     0.410               0.823               0.0              0.0   

   cat__City_Mumbai  cat__City_Pune  cat__Membership_Basic  \
0               0.0             0.0                    1.0   
1               1.0             0.0                    0.0   
2              

B. Combine the preprocessor and LinearRegression in a single Pipeline. Split the data into training and testing sets, evaluate the model using MAE and \(R^2\), and predict spending for new customers, including one whose city was not present during training.

In [4]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.DataFrame({
    "Age": [21, 25, 30, 35, 40, 45, 50, 28, 33, 38],
    "Annual_Income": [
        20000, 25000, 30000, 35000, 45000,
        50000, 60000, 28000, 40000, 48000
    ],
    "City": [
        "Delhi", "Mumbai", "Bhopal", "Delhi", "Pune",
        "Mumbai", "Bhopal", "Pune", "Delhi", "Mumbai"
    ],
    "Membership": [
        "Basic", "Silver", "Gold", "Basic", "Gold",
        "Silver", "Gold", "Basic", "Silver", "Gold"
    ],
    "Spending": [
        2000, 3000, 4500, 4000, 6000,
        6500, 8000, 3500, 5000, 7000
    ]
})

X = df[["Age", "Annual_Income", "City", "Membership"]]
y = df["Spending"]

numeric_features = ["Age", "Annual_Income"]
categorical_features = ["City", "Membership"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Actual spending:", y_test.to_list())
print("Predicted spending:", y_pred.round(2))

print("\nModel Evaluation")
print("MAE:", round(mae, 2))
print("R2 Score:", round(r2, 4))

new_customers = pd.DataFrame({
    "Age": [29, 42],
    "Annual_Income": [32000, 52000],
    "City": ["Delhi", "Jaipur"],
    "Membership": ["Silver", "Gold"]
})

new_predictions = model.predict(new_customers)

new_customers["Predicted_Spending"] = new_predictions.round(2)

print("\nNew Customer Predictions:")
print(new_customers.to_string(index=False))


Actual spending: [5000, 3000]
Predicted spending: [3637.5 3375. ]

Model Evaluation
MAE: 868.75
R2 Score: 0.0015

New Customer Predictions:
 Age  Annual_Income   City Membership  Predicted_Spending
  29          32000  Delhi     Silver             2787.50
  42          52000 Jaipur       Gold             6821.88


Question 3: Preventing Data Leakage with Pipelines
A financial institution wants to predict whether a customer will repay a loan. The dataset includes missing numerical values and categorical features.

import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Age": [25, 32, np.nan, 45, 50, 28, 38, np.nan, 42, 35],
    "Income": [
        30000, 45000, 50000, np.nan, 80000,
        35000, 60000, 55000, np.nan, 40000
    ],
    "Employment": [
        "Salaried", "Business", "Salaried", "Self-employed",
        "Business", "Salaried", np.nan, "Business",
        "Salaried", "Self-employed"
    ],
    "City": [
        "Delhi", "Mumbai", "Bhopal", "Delhi", "Pune",
        "Mumbai", "Delhi", "Pune", np.nan, "Bhopal"
    ],
    "Repaid_Loan": [1, 1, 0, 1, 1, 0, 1, 0, 1, 0]
})
A. Build a classification pipeline using ColumnTransformer, SimpleImputer, StandardScaler, OneHotEncoder, and LogisticRegression. Use median imputation for numerical features and most-frequent imputation for categorical features. Train and evaluate the model using accuracy and a confusion matrix.



In [5]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.DataFrame({
    "Age": [25, 32, np.nan, 45, 50, 28, 38, np.nan, 42, 35],
    "Income": [
        30000, 45000, 50000, np.nan, 80000,
        35000, 60000, 55000, np.nan, 40000
    ],
    "Employment": [
        "Salaried", "Business", "Salaried", "Self-employed",
        "Business", "Salaried", np.nan, "Business",
        "Salaried", "Self-employed"
    ],
    "City": [
        "Delhi", "Mumbai", "Bhopal", "Delhi", "Pune",
        "Mumbai", "Delhi", "Pune", np.nan, "Bhopal"
    ],
    "Repaid_Loan": [1, 1, 0, 1, 1, 0, 1, 0, 1, 0]
})

X = df.drop(columns=["Repaid_Loan"])
y = df["Repaid_Loan"]

numeric_features = ["Age", "Income"]
categorical_features = ["Employment", "City"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

print("Actual values:", y_test.to_list())
print("Predicted values:", y_pred.tolist())

print("\nAccuracy:", round(accuracy, 4))
print("\nConfusion Matrix (rows=actual, columns=predicted):")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred, labels=[0, 1], zero_division=0
))


Actual values: [0, 1]
Predicted values: [1, 1]

Accuracy: 0.5

Confusion Matrix (rows=actual, columns=predicted):
[[0 1]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



B. Explain and demonstrate the difference between fitting preprocessing on the entire dataset before splitting and fitting it only on training data through a pipeline. Identify which statistics or category information could leak from the test set and explain why this makes model evaluation less reliable.

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.DataFrame({
    "Age": [25, 32, np.nan, 45, 50, 28, 38, np.nan, 42, 35],
    "Income": [
        30000, 45000, 50000, np.nan, 80000,
        35000, 60000, 55000, np.nan, 40000
    ],
    "Employment": [
        "Salaried", "Business", "Salaried", "Self-employed",
        "Business", "Salaried", np.nan, "Business",
        "Salaried", "Self-employed"
    ],
    "City": [
        "Delhi", "Mumbai", "Bhopal", "Delhi", "Pune",
        "Mumbai", "Delhi", "Pune", np.nan, "Bhopal"
    ],
    "Repaid_Loan": [1, 1, 0, 1, 1, 0, 1, 0, 1, 0]
})

X = df.drop(columns="Repaid_Loan")
y = df["Repaid_Loan"]

num_cols = ["Age", "Income"]
cat_cols = ["Employment", "City"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X_all_transformed = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_all_transformed, y, test_size=0.2,
    random_state=42, stratify=y
)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

print("Accuracy with leakage:", accuracy_score(y_test, y_pred))

print("Median values learned from all data:",
      preprocessor.named_transformers_["num"]
      .named_steps["imputer"].statistics_)

print("Categories learned from all data:")
print(preprocessor.named_transformers_["cat"]
      .named_steps["encoder"].categories_)


Accuracy with leakage: 0.5
Median values learned from all data: [3.65e+01 4.75e+04]
Categories learned from all data:
[array(['Business', 'Salaried', 'Self-employed'], dtype=object), array(['Bhopal', 'Delhi', 'Mumbai', 'Pune'], dtype=object)]


Question 4: Machine Learning Pipeline for House Price Prediction
A real estate company wants to predict house prices using numerical property features.

import pandas as pd

df = pd.DataFrame({
    "Area": [600, 750, 800, 900, 1000, 1100, 1200, 1400, 1600, 1800,
             2000, 2200, 2400, 2600, 2800, 3000],
    "Bedrooms": [1, 1, 2, 2, 2, 3, 3, 3, 3, 4,
                 4, 4, 4, 5, 5, 5],
    "Bathrooms": [1, 1, 1, 2, 2, 2, 2, 2, 3, 3,
                  3, 3, 3, 4, 4, 4],
    "Age": [20, 18, 15, 12, 10, 9, 8, 7, 6, 5,
            4, 4, 3, 2, 2, 1],
    "Price": [1500000, 1700000, 1900000, 2200000, 2400000, 2700000,
              3000000, 3300000, 3600000, 4000000, 4300000, 4700000,
              5000000, 5500000, 5800000, 6200000]
})
A. Separate the input features and target, perform a train-test split, and build a pipeline containing StandardScaler and LinearRegression. Train the pipeline, predict prices for the test set, and calculate MAE and \(R^2\). Compare the predicted and actual prices in a DataFrame.



In [7]:

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.DataFrame({
    "Area": [600, 750, 800, 900, 1000, 1100, 1200, 1400,
            1600, 1800, 2000, 2200, 2400, 2600, 2800, 3000],
    "Bedrooms": [1, 1, 2, 2, 2, 3, 3, 3,3, 4, 4, 4, 4, 5, 5, 5],
    "Bathrooms": [1, 1, 1, 2, 2, 2, 2, 2,3, 3, 3, 3, 3, 4, 4, 4],
    "Age": [20, 18, 15, 12, 10, 9, 8, 7,6, 5, 4, 4, 3, 2, 2, 1],
    "Price": [
        1500000, 1700000, 1900000, 2200000,
        2400000, 2700000, 3000000, 3300000,
        3600000, 4000000, 4300000, 4700000,
        5000000, 5500000, 5800000, 6200000]
})

X = df[["Area", "Bedrooms", "Bathrooms", "Age"]]
y = df["Price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("R2 Score:", round(r2, 4))

comparison = pd.DataFrame({
    "Actual_Price": y_test,
    "Predicted_Price": y_pred.round(2)
})

comparison["Absolute_Error"] = (
    comparison["Actual_Price"] - comparison["Predicted_Price"]
).abs()

print("\nActual vs Predicted Prices:")
print(comparison.sort_index().round(2))


MAE: 76323.65
R2 Score: 0.9962

Actual vs Predicted Prices:
    Actual_Price  Predicted_Price  Absolute_Error
0        1500000       1307637.94       192362.06
1        1700000       1616531.85        83468.15
5        2700000       2721241.49        21241.49
14       5800000       5791777.08         8222.92


B. Use cross_val_score to evaluate the complete pipeline with five-fold cross-validation. Compare the cross-validation results with the single test-set result and explain why a single split can provide an unstable estimate when a dataset is small.

In [8]:

import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.DataFrame({
    "Area": [ 600, 750, 800, 900, 1000, 1100, 1200, 1400,
        1600, 1800, 2000, 2200, 2400, 2600, 2800, 3000],
    "Bedrooms": [1, 1, 2, 2, 2, 3, 3, 3,3, 4, 4, 4, 4, 5, 5, 5],
    "Bathrooms": [1, 1, 1, 2, 2, 2, 2, 2,3, 3, 3, 3, 3, 4, 4, 4],
    "Age": [20, 18, 15, 12, 10, 9, 8, 7,6, 5, 4, 4, 3, 2, 2, 1],
    "Price": [
        1500000, 1700000, 1900000, 2200000,
        2400000, 2700000, 3000000, 3300000,
        3600000, 4000000, 4300000, 4700000,
        5000000, 5500000, 5800000, 6200000
    ]
})
X = df[["Area", "Bedrooms", "Bathrooms", "Age"]]
y = df["Price"]
model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print("Single Test-Set Results")
print("MAE:", round(test_mae, 2))
print("R2:", round(test_r2, 4))

mae_scores = -cross_val_score(
    model, X, y, cv=5, scoring="neg_mean_absolute_error"
)

r2_scores = cross_val_score(
    model, X, y, cv=5, scoring="r2"
)

print("\nFive-Fold Cross-Validation")
print("MAE for each fold:", mae_scores.round(2))
print("Mean CV MAE:", round(mae_scores.mean(), 2))
print("Standard deviation of CV MAE:", round(mae_scores.std(), 2))

print("\nR2 for each fold:", r2_scores.round(4))
print("Mean CV R2:", round(r2_scores.mean(), 4))
print("Standard deviation of CV R2:", round(r2_scores.std(), 4))


Single Test-Set Results
MAE: 76323.65
R2: 0.9962

Five-Fold Cross-Validation
MAE for each fold: [101009.85  68235.09  46201.58  69502.06  81541.02]
Mean CV MAE: 73297.92
Standard deviation of CV MAE: 17945.2

R2 for each fold: [0.8057 0.873  0.9612 0.9294 0.9121]
Mean CV R2: 0.8963
Standard deviation of CV R2: 0.0535
